<a href="https://colab.research.google.com/github/Divya40789926/-Brewmind-ai/blob/main/fall_detection_demo_complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fall Detection Demo - Complete Notebook
Run every cell top to bottom, in order. Don't skip any.

In [12]:
# CELL 1: upload your project zip
from google.colab import files
print('Upload fall_detection_project.zip')
uploaded = files.upload()

Upload fall_detection_project.zip


Saving fall_detection_project (5).zip to fall_detection_project (5).zip


In [13]:
# CELL 2: extract it and move into the folder
import zipfile
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content')
%cd /content/fall_detection_project
!ls

/content/fall_detection_project
 config.yaml			   logs		     'results (1).zip'
 data				   models	      scripts
 DATA.md			   README.md	      src
'fall_detection_project (5).zip'   requirements.txt   yolov8n-pose.pt


In [14]:
# CELL 3: install dependencies
!pip install -q -r requirements.txt

In [15]:
# CELL 4: upload your ALREADY-TRAINED results zip (the one with fall_lstm_best.pt inside models/)
from google.colab import files
print('Upload your results zip (the one with the trained model)')
results_uploaded = files.upload()

Upload your results zip (the one with the trained model)


Saving results (1).zip to results (1) (1).zip


In [16]:
# CELL 5: extract just the model files from it
import zipfile, glob, shutil, os
results_zip_name = list(results_uploaded.keys())[0]
with zipfile.ZipFile(results_zip_name, 'r') as z:
    z.extractall('/content/results_extract')

os.makedirs('models', exist_ok=True)
ckpt_matches = glob.glob('/content/results_extract/**/fall_lstm_best.pt', recursive=True)
norm_matches = glob.glob('/content/results_extract/**/norm_stats.npz', recursive=True)
assert ckpt_matches, 'fall_lstm_best.pt not found in the uploaded zip!'
assert norm_matches, 'norm_stats.npz not found in the uploaded zip!'
shutil.copy(ckpt_matches[0], 'models/fall_lstm_best.pt')
shutil.copy(norm_matches[0], 'models/norm_stats.npz')
print('Copied checkpoint files:')
!ls -la models/

Copied checkpoint files:
total 256
drwxr-xr-x 2 root root   4096 Aug 30 22:18 .
drwxr-xr-x 7 root root   4096 Aug 30 22:23 ..
-rw-r--r-- 1 root root 247349 Aug 30 22:23 fall_lstm_best.pt
-rw-r--r-- 1 root root      0 Aug 30 22:21 .gitkeep
-rw-r--r-- 1 root root    828 Aug 30 22:23 norm_stats.npz


In [20]:
# Kaggle token (same one you used before, KGAT_...)
import os
from getpass import getpass
token = getpass('Paste your Kaggle API token (KGAT_...): ')
os.environ['KAGGLE_API_TOKEN'] = token
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(token)
!chmod 600 /root/.kaggle/access_token
!pip install -q -U kagglehub

Paste your Kaggle API token (KGAT_...): ··········


In [19]:
# CELL 6: Twilio credentials (paste when prompted -- hidden input, not saved to notebook file)
import os
from getpass import getpass
os.environ['TWILIO_ACCOUNT_SID'] = getpass('Twilio Account SID: ').strip()
os.environ['TWILIO_AUTH_TOKEN'] = getpass('Twilio Auth Token (from the AUTH TOKENS tab): ').strip()
print('Twilio credentials set for this session.')

Twilio Account SID: ··········
Twilio Auth Token (from the AUTH TOKENS tab): ··········
Twilio credentials set for this session.


In [22]:
!python scripts/download_datasets.py --config config.yaml --datasets urfd

2026-08-30 22:25:01,608 | download_datasets | INFO | Free space on target drive (/root): 87.3 GB
2026-08-30 22:25:01,611 | download_datasets | INFO | Downloading Kaggle dataset 'shahliza27/ur-fall-detection-dataset' (cached after first run)...
100% 4.18G/4.18G [00:49<00:00, 90.1MB/s]
Extracting files...
2026-08-30 22:26:25,603 | download_datasets | INFO | Kaggle dataset 'shahliza27/ur-fall-detection-dataset' available at: /root/.cache/kagglehub/datasets/shahliza27/ur-fall-detection-dataset/versions/1
2026-08-30 22:26:25,800 | download_datasets | WARNING | No video files (.mp4/.avi/.mov/.mkv) found in this URFD mirror. Running diagnostics on what was actually downloaded...
2026-08-30 22:26:25,965 | download_datasets | INFO | Downloaded dataset file type breakdown: {'.png': 11936}
2026-08-30 22:26:25,966 | download_datasets | INFO | Sample of first 25 files:
2026-08-30 22:26:25,966 | download_datasets | INFO |   UR_fall_detection_dataset_cam0_rgb/fall-08-cam0-rgb/fall-08-cam0-rgb-057.png

In [23]:
import os
print(os.listdir('data/raw/fall'))

['fall-29-cam0-rgb.mp4', 'fall-26-cam0-rgb.mp4', 'fall-27-cam0-rgb.mp4', 'fall-24-cam0-rgb.mp4', 'fall-09-cam0-rgb.mp4', 'fall-10-cam0-rgb.mp4', 'fall-20-cam0-rgb.mp4', 'fall-28-cam0-rgb.mp4', 'fall-23-cam0-rgb.mp4', 'fall-15-cam0-rgb.mp4', 'fall-13-cam0-rgb.mp4', 'fall-12-cam0-rgb.mp4', 'fall-01-cam0-rgb.mp4', 'fall-18-cam0-rgb.mp4', 'fall-30-cam0-rgb.mp4', 'fall-06-cam0-rgb.mp4', 'fall-17-cam0-rgb.mp4', 'fall-19-cam0-rgb.mp4', 'fall-14-cam0-rgb.mp4', 'fall-25-cam0-rgb.mp4', 'fall-04-cam0-rgb.mp4', 'fall-05-cam0-rgb.mp4', 'fall-11-cam0-rgb.mp4', 'fall-02-cam0-rgb.mp4', '.gitkeep', 'fall-16-cam0-rgb.mp4', 'fall-22-cam0-rgb.mp4', 'fall-07-cam0-rgb.mp4', 'fall-03-cam0-rgb.mp4', 'fall-08-cam0-rgb.mp4', 'fall-21-cam0-rgb.mp4']


In [24]:
# CELL 7: set the non-secret config values -- EDIT THE THREE LINES BELOW WITH YOUR REAL VALUES FIRST
import yaml
with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['alerts']['twilio']['from_number'] = '+17372508034'      # <-- your Twilio trial number
cfg['alerts']['twilio']['to_numbers'] = ['+916363969881']     # <-- your verified personal number
cfg['pose']['device'] = 'cpu'                                  # no GPU needed/available right now
cfg['realtime']['fall_confidence_threshold'] = 0.5

with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f)
print('config.yaml updated.')

config.yaml updated.


In [25]:
# CELL 8: list your fall videos so you can pick one
import os
print(os.listdir('data/raw/fall'))

['fall-29-cam0-rgb.mp4', 'fall-26-cam0-rgb.mp4', 'fall-27-cam0-rgb.mp4', 'fall-24-cam0-rgb.mp4', 'fall-09-cam0-rgb.mp4', 'fall-10-cam0-rgb.mp4', 'fall-20-cam0-rgb.mp4', 'fall-28-cam0-rgb.mp4', 'fall-23-cam0-rgb.mp4', 'fall-15-cam0-rgb.mp4', 'fall-13-cam0-rgb.mp4', 'fall-12-cam0-rgb.mp4', 'fall-01-cam0-rgb.mp4', 'fall-18-cam0-rgb.mp4', 'fall-30-cam0-rgb.mp4', 'fall-06-cam0-rgb.mp4', 'fall-17-cam0-rgb.mp4', 'fall-19-cam0-rgb.mp4', 'fall-14-cam0-rgb.mp4', 'fall-25-cam0-rgb.mp4', 'fall-04-cam0-rgb.mp4', 'fall-05-cam0-rgb.mp4', 'fall-11-cam0-rgb.mp4', 'fall-02-cam0-rgb.mp4', '.gitkeep', 'fall-16-cam0-rgb.mp4', 'fall-22-cam0-rgb.mp4', 'fall-07-cam0-rgb.mp4', 'fall-03-cam0-rgb.mp4', 'fall-08-cam0-rgb.mp4', 'fall-21-cam0-rgb.mp4']


In [26]:
# CELL 9: pick the video -- EDIT THE FILENAME BELOW to one from CELL 8's output
import yaml
with open('config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['realtime']['camera_source'] = 'data/raw/fall/fall-03-cam0-rgb.mp4'  # <-- EDIT THIS
with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f)
print('camera source set to', cfg['realtime']['camera_source'])

camera source set to data/raw/fall/fall-03-cam0-rgb.mp4


In [27]:
# CELL 10: THE MAIN DEMO -- runs inference, draws skeleton + fall label, saves video, sends real SMS, downloads the file
import cv2, numpy as np, torch, yaml
from collections import deque
from pathlib import Path
from src.model import FallLSTM
from src.feature_engineering import _angle, _bbox_aspect_ratio
from src.alert import AlertManager
from ultralytics import YOLO

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

ckpt_path = Path(cfg['paths']['checkpoint_dir']) / 'fall_lstm_best.pt'
assert ckpt_path.exists(), f'Checkpoint missing at {ckpt_path} -- re-run CELL 5'

device = torch.device('cpu')
ckpt = torch.load(ckpt_path, map_location=device)
model = FallLSTM(
    input_size=ckpt['input_size'],
    hidden_size=ckpt['config_model']['hidden_size'],
    num_layers=ckpt['config_model']['num_layers'],
    dropout=ckpt['config_model']['dropout'],
    bidirectional=ckpt['config_model']['bidirectional'],
).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

norm = np.load(Path(cfg['paths']['checkpoint_dir']) / 'norm_stats.npz')
mean, std = norm['mean'], norm['std']

pose_model = YOLO(cfg['pose']['model_name'])
alert_manager = AlertManager(cfg)

video_path = cfg['realtime']['camera_source']
window_size = cfg['features']['window_size']
triplets = cfg['features']['joint_angle_triplets']
threshold = cfg['realtime']['fall_confidence_threshold']

cap = cv2.VideoCapture(video_path)
assert cap.isOpened(), f'Could not open video at {video_path} -- check CELL 9'
w, h, fps = int(cap.get(3)), int(cap.get(4)), cap.get(5) or 25

out_avi = '/content/fall_detection_project/demo_output.avi'
writer = cv2.VideoWriter(out_avi, cv2.VideoWriter_fourcc(*'XVID'), fps, (w, h))
if not writer.isOpened():
    writer = cv2.VideoWriter(out_avi, cv2.VideoWriter_fourcc(*'MJPG'), fps, (w, h))
print('Writer opened:', writer.isOpened())
assert writer.isOpened(), 'VideoWriter failed to open with both XVID and MJPG codecs.'

feature_buffer = deque(maxlen=window_size)
prev_xy = np.zeros((17, 2), dtype=np.float32)
fired = False
frame_count = 0
max_fall_prob = 0.0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1

    results = pose_model(frame, verbose=False, conf=cfg['pose']['conf_threshold'])
    label, color = '', (255, 255, 255)

    if len(results) and results[0].keypoints is not None and len(results[0].keypoints.data) > 0:
        kpts_all = results[0].keypoints.data.cpu().numpy()
        confs = results[0].boxes.conf.cpu().numpy() if len(results[0].boxes.conf) else np.array([0])
        best_idx = int(np.argmax(confs))
        curr_xy = kpts_all[best_idx][:, :2]
        frame = results[0].plot(img=frame)
    else:
        curr_xy = np.zeros((17, 2), dtype=np.float32)

    velocity = (curr_xy - prev_xy).reshape(-1)
    angles = np.array([_angle(curr_xy[a], curr_xy[b], curr_xy[c]) for a, b, c in triplets], dtype=np.float32)
    aspect = np.array([_bbox_aspect_ratio(curr_xy)], dtype=np.float32)
    feat = np.concatenate([velocity, angles, aspect]).astype(np.float32)
    prev_xy = curr_xy
    feature_buffer.append(feat)

    if len(feature_buffer) == window_size:
        xw = (np.stack(feature_buffer) - mean) / std
        x = torch.tensor(xw, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            prob = torch.sigmoid(model(x)).item()
        max_fall_prob = max(max_fall_prob, prob)
        if prob >= threshold:
            label, color = f'FALL DETECTED ({prob*100:.0f}%)', (0, 0, 255)
            if not fired:
                alert_manager.send_fall_alert(f'Fall detected with {prob*100:.0f}% confidence. Please check immediately.')
                fired = True
        else:
            label, color = f'Normal ({(1-prob)*100:.0f}%)', (0, 200, 0)

    if label:
        cv2.rectangle(frame, (10, 10), (10 + 9 * len(label), 45), (0, 0, 0), -1)
        cv2.putText(frame, label, (18, 38), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

    writer.write(frame)

cap.release()
writer.release()
print(f'DONE - {frame_count} frames processed. Max fall probability seen: {max_fall_prob:.2f}')
print(f'Alert fired: {fired}')
!ls -la /content/fall_detection_project/demo_output.avi

!ffmpeg -y -i /content/fall_detection_project/demo_output.avi -vcodec libx264 -pix_fmt yuv420p /content/fall_detection_project/demo_output_web.mp4
!ls -la /content/fall_detection_project/demo_output_web.mp4

from google.colab import files
files.download('/content/fall_detection_project/demo_output_web.mp4')

2026-08-30 22:33:00,459 | alert | WARNING | Firebase not configured ([Errno 2] No such file or directory: 'firebase_credentials.json'); push alerts disabled.


Writer opened: True


2026-08-30 22:33:14,079 | alert | ERROR | Failed to send SMS to +916363969881: HTTP 400 error: Unable to create record: Invalid template name. Trial accounts can only use predefined SMS templates.
ERROR:alert:Failed to send SMS to +916363969881: HTTP 400 error: Unable to create record: Invalid template name. Trial accounts can only use predefined SMS templates.
2026-08-30 22:33:14,081 | alert | INFO | [Push skipped - not configured] Fall detected with 55% confidence. Please check immediately.
INFO:alert:[Push skipped - not configured] Fall detected with 55% confidence. Please check immediately.


DONE - 215 frames processed. Max fall probability seen: 1.00
Alert fired: True
-rw-r--r-- 1 root root 3772370 Aug 30 22:33 /content/fall_detection_project/demo_output.avi
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsna

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
from src.alert import AlertManager

alert_manager = AlertManager(cfg)

try:
    result = alert_manager.send_fall_alert(
        "TEST ALERT: Twilio SMS is working!"
    )
    print("SUCCESS:", result)
except Exception as e:
    print("ERROR:", e)

2026-08-30 22:42:32,362 | alert | WARNING | Firebase not configured ([Errno 2] No such file or directory: 'firebase_credentials.json'); push alerts disabled.
2026-08-30 22:42:32,559 | alert | ERROR | Failed to send SMS to +916363969881: HTTP 400 error: Unable to create record: Invalid template name. Trial accounts can only use predefined SMS templates.
ERROR:alert:Failed to send SMS to +916363969881: HTTP 400 error: Unable to create record: Invalid template name. Trial accounts can only use predefined SMS templates.
2026-08-30 22:42:32,561 | alert | INFO | [Push skipped - not configured] TEST ALERT: Twilio SMS is working!
INFO:alert:[Push skipped - not configured] TEST ALERT: Twilio SMS is working!


SUCCESS: None


In [28]:
# CELL 11 (optional): play it inline right here too, if the download didn't auto-trigger
from IPython.display import Video
Video('/content/fall_detection_project/demo_output_web.mp4', embed=True, width=640)